# Exercise 19 - PCA, t-SNE, and K-Means

Estimated time: **35-40 minutes**

The goal of this exercise is to get a feel for the PCA, t-SNE and K-Means algorithms, and to apply PCA to the California Housing datasets to perform dimensionality reduction.

Principle Component Analysis (PCA) and t-distributed Stochastic Neighbor Embedding (t-SNE) are two popular techniques for dimensionality reduction and visualization. They are both examples of unsupervised learning algorithms, able to identify regularities in an unlabeled dataset.

Informally speaking, given a high-dimensional dataset, PCA finds the components of that dataset that represent the most variability within the dataset. For example, given a 1000-dimensional dataset, PCA could identify the 2 dimensions that vary the most and project the entire dataset onto those 2 dimensions. The resultant projection could be used as a 2-D plot, making PCA useful for visualization. Although the components identified by PCA could be axes, in general they will not be axis but are arbitrary vectors, the particular vectors in the high-dimensional space that represent the greatest degree of variability in the dataset and thus characterize the data most succinctly.

PCA is not just used for visualization. PCA can be used to reduce the dimensionality of a dataset prior to machine learning. In effect, PCA can be used to reduce the number of features in a high-dimensional dataset, leaving only the features that have the biggest impact. This can reduce the network size and the number of parameters to be learned. When used for visualization, PCA might be used to reduce a 1,000-dimensional dataset to just 2 or 3 dimensions for plotting. When used for dimensionality reduction prior to learning, PCA might reduce a 10-dimensional dataset to just 8 dimensions by eliminating the features that have least impact when differentiating between samples (remembering that each principle component will actually be a combination of features rather than a single feature, so each of the 8 dimensions that remain would be some linear combination of the original 10 dimensions).

t-SNE has become a very popular algorithm for visualizing very-high-dimensional machine learning datasets as a scatterplot in 2 or 3 dimensions. Its effect is to place similar high-dimensional objects close together in a plot and to place dissimilar objects far apart in a plot.

- You can use **CPU** for this exercise

## PCA and t-SNE on The MNIST Dataset

Here we read in the MNIST dataset then plot a subset of the characters in 2-dimensions using PCA and then using t-SNE.

First run the code below to read in the MNIST dataset and select a subset of the data samples.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

(images, labels), _ = mnist.load_data()

N = 2000
images = images[:N,:].reshape(-1,784)
labels = labels[:N]

Here we are using the [scikit-learn](http://scikit-learn.org) Python package, which gives us PCA and t-SNE algorithms out-of-the-box. Run the following code and take a look at the PCA and t-SNE visualizations. Remember that these algorithms are just looking at the images. The algorithms are not being given the digit that each image represents. The labels 0-9 are being used in the plots but are not available to the algorithms. You can see why t-SNE is popular!

In [ ]:
from sklearn.decomposition import PCA
from sklearn import manifold

X = images
y = labels

def plot_embedding(X, title=None):
    x_min, x_max = np.min(X, 0), np.max(X, 0)
    X = (X - x_min) / (x_max - x_min)

    plt.figure(1, figsize=(15,15))
    for i in range(X.shape[0]):
        plt.text(X[i, 0], X[i, 1], str(labels[i]),
                 color=plt.cm.Set1(y[i] / 10.),
                 fontdict={'weight': 'bold', 'size': 9})
    plt.xticks([]), plt.yticks([])
    if title is not None:
        plt.title(title)

pca = PCA(n_components=2)
pca.fit(X)
X_pca = pca.transform(X)

# The following would be equivalent
#X_pca = decomposition.TruncatedSVD(n_components=2).fit_transform(X)

plot_embedding(X_pca, "2-D PCA projection of the digits")
plt.show()

tsne = manifold.TSNE(n_components=2, init='pca', random_state=0)
X_tsne = tsne.fit_transform(X)

plot_embedding(X_tsne, "t-SNE embedding of the digits")
plt.show()

## K-Means on The MNIST Dataset

K-Means is one of the simplest unsupervised clustering algorithm, simpler than PCA or t-SNE. It clusters samples according to their Euclidean distance from K chosen centroids, moving the centroids around to minimize the total distance. We give K-Means a try on the MNIST dataset just to see what it can achieve:

In [ ]:
from sklearn.cluster import KMeans

# Pick 10 clusters because we know there are 10 labels
kmeans = KMeans(n_clusters=10, random_state=42)
labels = kmeans.fit_predict(X)

# Figure out how to re-order the labels
print('Re-order the clusters to best match the original labels:')
print('Cluster -> label')
for i in range(10):
    counts = np.bincount(y[labels==i])
    n = np.argmax(counts)
    print(i, '->', n, counts)

# Do the re-ordering
labels = np.array(tuple(map(lambda i: [7, 9, 5, 4, 0, 1, 3, 8, 2, 6][i], labels)))

from sklearn.metrics import confusion_matrix, accuracy_score

print(f'\nAccuracy score after re-ordering = {accuracy_score(y, labels):4.2f}\n')
print(confusion_matrix(y, labels))

K-Means is able to achieve clusters with a 52% match to the originals without seeing the original labels. You can see the strong concentration of values on the diagonal of the confusion matrix, although you can also see that many 4's, 5's and 9's are being clustered wrongly, and that the '9's cluster does not actually contain any true 9 labels. If you inspect the PCA and t-SNE plots above you will see that this is perhaps not really surprising.

## PCA on the California Housing Dataset

In the final part of this exercise you have some work to do - apply PCA to the California Housing dataset.

First read in the dataset.

In [ ]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
print(dir(housing))
print(housing.data.shape)
print(housing.target.shape)
print(housing.feature_names)

Try linear regression out-of-the-box to create a model that would be capable of predicting a house price given a novel set of features (crime rate, property tax, and so on) it had never seen before:

In [ ]:
X, y = housing.data, housing.target

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=1)

from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print(f'Linear regression score (training data) = {regressor.score(X_train, y_train):4.2f}')
print(f'Linear regression score (test data)     = {regressor.score(X_test, y_test):4.2f}')

Now calculate the principal components of the California Housing dataset and print the explained variance ratio for each principal component:

In [ ]:
#

Apply PCA dimensionality reduction to the dataset for 1, 2, 3 up to the maximum number of principle components and do a linear regression in each case, then plot the score from the linear regression against the number of components used.

In [ ]:
train_score = []
test_score = []

for n in range(1, m):
    #
    #
    #
    train_score.append(
    test_score.append(

plt.figure(1, figsize=(9,6))
plt.plot(range(1,m), train_score, label='Training score')
plt.plot(range(1,m), test_score, label='Test score')
plt.legend(loc="best")
plt.xlabel('Number of principal components')
plt.show()

#### Solution

If you want some help with the answer, you can look at our answer.  Copy and paste the code in a new code cell to try it out. 

<details>
    <summary> Here is our answer </summary>
    
    from sklearn.datasets import fetch_california_housing
    housing = fetch_california_housing()
    X, y = housing.data, housing.target

    #from sklearn.preprocessing import StandardScaler
    #StandardScaler(copy=False).fit_transform(X)

    #from sklearn.preprocessing import RobustScaler
    #RobustScaler(copy=False).fit_transform(X)

    from sklearn.decomposition import PCA
    pca = PCA(n_components=8)
    pca.fit(X)

    # Show the % of the variance explained by each principal component
    for i in range(pca.n_components):
        print(f'{i+1:2} {100*pca.explained_variance_ratio_[i]:6.3f}%')

    train_score = []
    test_score = []
    m = 9
    for n in range(1,m):
        pca = PCA(n_components=n)
        pca.fit(X)
        X_pca = pca.transform(X)

        X_train, X_test, y_train, y_test = train_test_split(X_pca, y, train_size=0.75, random_state=1)

        regressor = LinearRegression()
        regressor.fit(X_train, y_train)

        train_score.append(regressor.score(X_train, y_train))
        test_score.append(regressor.score(X_test, y_test))

    plt.figure(1, figsize=(9,6))
    plt.plot(range(1,m), train_score, label='Training score')
    plt.plot(range(1,m), test_score, label='Test score')
    plt.legend(loc="best")
    plt.xlabel('Number of principal components')
    plt.show()

    
</details>